## 案例: 演示通过 朴素贝叶斯算法 实现  商品评论情感分析, 
## 即: 好评, 差评...

朴素贝叶斯介绍:

    概述:
        贝叶斯: 仅仅依赖 概率 就可以进行分类的 一种机器学习算法.
        朴素:   不考虑特征之间的关联性, 即: 特征间都是相互独立的.
            原始:  P(AB) = P(A) * P(B|A) = P(B) * P(A|B)
            加入朴素后: P(AB) = P(A) * P(B)
    细节:
        因为我们分词要用到 jieba分词器, 记得先装一下, 例如: pip install jieba

#### 导包

In [1]:
import numpy as np                  # 数学计算包
import pandas as pd                 # 数据处理包
import matplotlib.pyplot as plt     # 画图包
import jieba                        # 分词包
from sklearn.feature_extraction.text import CountVectorizer # 词频统计包, 把评论内容 转成 词频矩阵.
from sklearn.metrics import accuracy_score
from sklearn.naive_bayes import MultinomialNB               # 朴素贝叶斯对象

#### 1. 读取文件，获取到原始数据

In [2]:
df = pd.read_csv('./data/书籍评价1.csv', encoding='utf-8')
df.info()
print(df)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 3 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   Unnamed: 0  500 non-null    int64 
 1   内容          500 non-null    object
 2   评价          500 non-null    object
dtypes: int64(1), object(2)
memory usage: 11.8+ KB
     Unnamed: 0                                                内容  评价
0             0                               代码示例都能直接跑，新手跟着练上手极快  好评
1             1                               通篇理论堆砌，连一行完整实战代码都没有  差评
2             2                               语言特别通俗，完全不用啃晦涩的专业术语  好评
3             3                                知识点跳得太快，零基础根本跟不上节奏  差评
4             4                               章节逻辑很顺，从基础到实战过渡特别自然  好评
..          ...                                               ...  ..
495         495  这本书节奏过快且跳步严重，零基础根本无法跟上学习进度，内容还老套落后，和当前编程学习需求完全脱节  差评
496         496  这本书重点标注清晰且课后练习设计合理，能有效检验学习效果，通俗易懂的讲解让编程入门变得简单又轻松  好评
497         497

#### 2. 数据预处理

In [3]:
# 2.1 添加labels列, 充当: 标签列.  好评 -> 1, 差评 -> 0
df['labels'] = np.where(df['评价'] == '好评', 1, 0)
# df.info()
# print(df)

# 2.2 抽取 labels列, 作为: 标签.
y = df['labels']

# 2.3 演示 jieba 分词
print(jieba.lcut('好好学习, 天天向上! 我爱你你爱我, 蜜雪冰城甜蜜蜜! 小明骑车, 一把把把把住了.'))

# 2.4 对用户的评论信息, 做切词.
# 数据格式: [[第1条评论切词1, 切词2, 切词3...], [第2条评论切词1, 切词2, 切词3...], ...]
comment_list = [','.join(jieba.lcut(line)) for line in df['内容']]
# 数据格式: ['第1条评论切词1, 切词2, 切词3...', '第2条评论切词1, 切词2, 切词3...', ...]
print(comment_list)

# 演示字符串的 join()函数用法.
# my_list = ['aa', 'bb', 'cc']
# print(','.join(my_list))

# 2.5 加载 停用词列表, 即: 里边记录的词, 不需要参与模型训练, 预测, 要被删除的词, 例如: 的, 啊, 哈, 从, 都...
with open('./data/stopwords.txt', 'r', encoding='utf-8') as src_f:
    # 2.5.1 一次读取所有的行
    # src_f.readlines()：文件对象的内置方法，作用是一次性读取文件的所有行，返回一个列表
    stopwords_list = src_f.readlines()
    # 2.5.2 删除最后的 '\n'
    # line.strip()：字符串的内置方法，作用是删除字符串两端的空白字符（包括换行符\n、空格、制表符\t等）
    stopwords_list = [line.strip() for line in stopwords_list]
    # 2.5.3 对 停用词列表去重.
    # 利用集合（set）的特性去重的语法，核心逻辑是 “列表→集合→列表”
    # set(stopwords_list)：将列表转换成集合，集合的特性是元素唯一、无重复，所以会自动去掉列表中的重复值
    # list(...)：将去重后的集合转回列表（因为集合不能按索引访问，列表更常用）
    stopwords_list = list(set(stopwords_list))
    # print(stopwords_list)

# 2.6 创建向量化对象, 从 评论切词列表(comment_list) 中 删除 停用词, 并且统计词频(单词矩阵).
transfer = CountVectorizer(stop_words=stopwords_list)   # 参数: 停用词列表.
# 2.7 统计词频矩阵, 先训练, 后转换, 在转数组.
# transfer.fit(comment_list)
# x的格式: [[第1条评论的切词分布, 有就是1, 没有就是0], [第2条评论的切词分布, 有就是1, 没有就是0], ...]
# x = transfer.transform(comment_list).toarray()
x = transfer.fit_transform(comment_list).toarray()
print(x)

# 2.8 看一下 我们评论, 切词, 且删除 停用词后, 一共剩下多少个词了.
print(transfer.get_feature_names_out())
print(len(transfer.get_feature_names_out()))    # 评论切词且删除停用词后, 一共剩下多少个词了.

# 2.9 
x_train = x[:480]
y_train = y[:480]

x_test = x[480:]
y_test = y[480:]

Building prefix dict from the default dictionary ...
Loading model from cache C:\Users\35399\AppData\Local\Temp\jieba.cache
Loading model cost 0.943 seconds.
Prefix dict has been built successfully.


['好好学习', ',', ' ', '天天向上', '!', ' ', '我爱你', '你', '爱', '我', ',', ' ', '蜜雪', '冰城', '甜蜜蜜', '!', ' ', '小明', '骑车', ',', ' ', '一把', '把', '把', '把住', '了', '.']
['代码,示例,都,能,直接,跑,，,新手,跟着,练上,手,极快', '通篇,理论,堆砌,，,连,一行,完整,实战,代码,都,没有', '语言,特别,通俗,，,完全,不用,啃,晦涩,的,专业术语', '知识点,跳得,太快,，,零,基础,根本,跟不上,节奏', '章节,逻辑,很,顺,，,从,基础,到,实战,过渡,特别,自然', '内容,又,水,又,浅,，,有,一点,编程,基础,的,看,了,纯,浪费时间', '小,项目,设计,得,很,用心,，,学完,能,独立,做出,简单,作品', '文字,生硬,难懂,，,越读,越,懵,，,完全,不,适合,自学', '重点,标注,得,很,清楚,，,新手,一眼,就,能,抓住,核心,知识', '全是,废话,凑,字数,，,真正,有用,的,知识点,寥寥无几', '常见,坑点,都,提前,提醒,，,帮,新手,少,走,超多,弯路', '内容,陈旧,过时,，,和,现在,的,编程,环境,完全,脱节', '零,基础,也,能,轻松,看,懂,，,彻底消除,对,编程,的,恐惧', '讲解,含糊不清,，,关键步骤,直接,跳过,不,解释', '结构,安排,合理,，,自学,跟着,走,完全,不用,别人,指导', '逻辑,混乱,东拼西凑,，,越学越,没有,头绪', '干货,特别,足,，,没有,多余,内容,，,每,一页,都,有,收获', '实用性,极差,，,学完,依旧,不会,写,独立程序', '特别,适合,纯小白,打基础,，,入门,首选,的,良心,书', '排版,杂乱,字体,不清,，,阅读,体验,差到,不想,看', '由浅入深,讲解,透彻,，,看,完能,建立,完整,编程,思维', '内容,太,单薄,，,知识点,少得,可怜,，,完全,不值,这个,价', '案例,贴近,实战,，,学完,知识,能,直接,用到,实际,编程,里', '讲解,敷衍,潦草,，,很多,知识点,只,讲,一半,就,没,下文', '通俗易懂,不,啰嗦,，,零,基础,入门,效率,特别,高', '节奏,拖沓,重复,，,简

#### 3. 特征工程（略）

#### 4. 模型训练

In [4]:
estimator = MultinomialNB()     # 创建 朴素贝叶斯模型对象.
estimator.fit(x_train, y_train)

,alpha,1.0
,force_alpha,True
,fit_prior,True
,class_prior,None


#### 5. 模型预测

In [5]:
y_pred = estimator.predict(x_test)
print(y_test)
print(f'模型预测结果: {y_pred}')

480    1
481    0
482    1
483    0
484    1
485    0
486    1
487    0
488    1
489    0
490    1
491    0
492    1
493    0
494    1
495    0
496    1
497    0
498    1
499    0
Name: labels, dtype: int64
模型预测结果: [1 0 1 0 1 0 1 0 1 0 1 0 1 0 1 0 1 0 1 0]
